# 01 · Vibes Lite setup and overview

This notebook is the entry point of the platform: it imports the processing
engines, prints what the deployment can do, and makes sure a demo dataset
exists.

**Everything here runs on the labelled synthetic demo scene.** It is generated
by Vibes Lite (`data.generate_synthetic_scene`) and tagged `SYNTHETIC=true`, so
the numbers you see are modelled reflectance, not real satellite measurements.
Swap in your own GeoTIFF + GeoJSON at any point - the engines do not care.

In [ ]:
%matplotlib inline
# --- Vibes Lite bootstrap -----------------------------------------------------
# These notebooks drive the processing engines directly (no HTTP layer needed).
import sys
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import config, data, geospatial, preprocessing, indices, analysis
import opencv as vl_cv          # Vibes Lite's own engine (it wraps cv2 internally)
import segmentation, timeseries, visualization, storage

PATHS = config.ensure_directories()
NOTEBOOK_OUT = PATHS["outputs"] / "notebooks"
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)

print("project root :", ROOT)
print("raw data     :", PATHS["raw_data"].relative_to(ROOT))
print("notebook out :", NOTEBOOK_OUT.relative_to(ROOT))

## What this deployment can do

Configuration, indices, OpenCV operations and segmentation methods, all read from `config.yaml`.

In [ ]:
info = config.project_info()
keys = ("name", "version", "default_index", "crop_health_thresholds", "opencv_enabled", "visualization_enabled")
print(json.dumps({key: info[key] for key in keys}, indent=2))
print("\nindices      :", ", ".join(sorted(indices.INDEX_REGISTRY)))
print("opencv ops   :", ", ".join(vl_cv.available_operations()))
print("segmentation :", ", ".join(segmentation.available_methods()))

## Demo data

The scene contains four plots (winter wheat, maize, sunflower and a nutrient trial plot), a nutrient-stress patch and a bare-soil strip, so NDVI, health zones and weak-vegetation detection all have something to find.

In [ ]:
DEMO_NAME = "synthetic_fields"


def demo_scene(name: str = DEMO_NAME, size: int = 900):
    """Return (raster, aoi, csv) for the labelled synthetic demo scene.

    The scene is only generated when missing. It is explicitly synthetic - its
    GeoTIFF tags carry SYNTHETIC=true - so nothing here pretends to be a real
    satellite acquisition.
    """
    raster = PATHS["raw_data"] / f"{name}.tif"
    aoi = PATHS["raw_data"] / f"{name}.geojson"
    table = PATHS["raw_data"] / f"{name}_ndvi_history.csv"
    if not raster.exists() or not aoi.exists():
        scene = data.generate_synthetic_scene(name=name, size=size, overwrite=True)
        return scene["paths"]["raster"], scene["paths"]["aoi"], scene["paths"]["table"]
    return raster, aoi, table


RASTER, AOI_FILE, HISTORY_CSV = demo_scene()
print("raster :", RASTER.name)
print("aoi    :", AOI_FILE.name)
print("history:", HISTORY_CSV.name)

In [ ]:
datasets = pd.DataFrame(data.list_datasets())
print(datasets[["name", "type", "band_count", "size_mb", "synthetic"]].to_string(index=False))

## Raster metadata and validation

`get_raster_metadata` is the read-only inspection path; `validate_raster` adds the upload guard rails (size, CRS, band count, readability).

In [ ]:
metadata = data.get_raster_metadata(RASTER)
print("keys:", sorted(metadata))
print(json.dumps({
    "band_count": metadata["band_count"],
    "band_descriptions": metadata["band_descriptions"],
    "dtypes": metadata["dtypes"],
    "size_px": [metadata["width"], metadata["height"]],
    "resolution": metadata["resolution"],
    "area_ha": metadata["area_ha"],
    "is_georeferenced": metadata["is_georeferenced"],
    "synthetic": metadata["synthetic"],
    "acquired": metadata["tags"].get("ACQUIRED"),
}, indent=2, default=str))
print("crs:", metadata["crs"])

In [ ]:
validation = data.validate_raster(RASTER)
print("validation keys:", sorted(validation))
print("warnings       :", validation["warnings"] or "none")

band = data.load_single_band(RASTER, 4)
nir = np.asarray(band["array"], dtype="float32")
print("\nNIR band (4) shape:", nir.shape, "| min:", float(nir.min()), "| max:", float(nir.max()))

## Next steps

* **02_load_aoi.ipynb** - field boundaries, CRS handling, AOI/raster overlap
* **03_ndvi.ipynb** - NDVI from RED + NIR with statistics
* **04_crop_health.ipynb** - configurable health classes and weak zones
* **05_opencv.ipynb** - the OpenCV engine (masks, morphology, contours)
* **06_segmentation.ipynb** - classical segmentation, AI-ready registry
* **07_time_series.ipynb** - NDVI over time, trend and change detection
* **08_final_demo.ipynb** - the whole pipeline end to end, saved as a result